# noise_reachability step 18: constancy or lower noise diversity? (mechanism M1; Kaggle, 2 x T4)

Step 18 of `noise_reachability/README.md`. No training. The base policy on 8 resistant states with
constant noise (`repeat`), a slowly drifting noise (rho 0.9), half-temperature sampling (sigma 0.5) and the reported
processes: 2896 episodes in 296 jobs of at most 10, after a 16-episode harness.

**Needs no input on the first session. Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing
needs editing. The wrapper checks and the harness run first; a failure stops the run.

**This run needs two sessions (about 18 h of episodes; an estimate).** When step 12 says work remains, add this
version's output as input (**Add Input > Your Work**, only the latest step 18 version) and Save & Run All again. Step
6 must then print `RESUMING`.

Download from the Output tab: `step18_results.zip`.

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "5554dafbf35fa86202816402d2ebaffb4bac84f3"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 18; nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [0, 4, 6, 7]                 # the wrapper checks run on each
N_ACTION_STEPS = 10
STATES = [[0, 8], [0, 6], [4, 26], [4, 11], [6, 12], [6, 8], [7, 33], [7, 47]]
CONDITIONS = {'repeat': ['repeat', None, None], 'ar_repeat_rho0.9': ['ar_repeat', 0.9, None], 'scaled_sigma0.5': ['scaled', None, 0.5], 'ar_repeat_rho0.5': ['ar_repeat', 0.5, None], 'ar_repeat_rho0': ['ar_repeat', 0.0, None], 'scaled_sigma1': ['scaled', None, 1.0], 'zero': ['zero', None, None]}   # label: [family, rho, sigma]
FLAGS = {'repeat': '--family repeat', 'ar_repeat_rho0.9': '--family ar_repeat --rho 0.9', 'scaled_sigma0.5': '--family scaled --sigma 0.5', 'ar_repeat_rho0.5': '--family ar_repeat --rho 0.5', 'ar_repeat_rho0': '--family ar_repeat --rho 0', 'scaled_sigma1': '--family scaled --sigma 1', 'zero': '--family zero'}
BLOCKS = {'repeat': {'0/8': ['20000-20009', '20010-20019', '20020-20029', '20030-20039', '20040-20049', '20050-20059', '20060-20069', '20070-20079', '20080-20089', '20090-20099'], '0/6': ['20200-20209', '20210-20219', '20220-20229', '20230-20239', '20240-20249', '20250-20259', '20260-20269', '20270-20279', '20280-20289', '20290-20299'], '4/26': ['20400-20409', '20410-20419', '20420-20429', '20430-20439', '20440-20449', '20450-20459', '20460-20469', '20470-20479', '20480-20489', '20490-20499'], '4/11': ['20600-20609', '20610-20619', '20620-20629', '20630-20639', '20640-20649', '20650-20659', '20660-20669', '20670-20679', '20680-20689', '20690-20699'], '6/12': ['20800-20809', '20810-20819', '20820-20829', '20830-20839', '20840-20849', '20850-20859', '20860-20869', '20870-20879', '20880-20889', '20890-20899'], '6/8': ['21000-21009', '21010-21019', '21020-21029', '21030-21039', '21040-21049', '21050-21059', '21060-21069', '21070-21079', '21080-21089', '21090-21099'], '7/33': ['21200-21209', '21210-21219', '21220-21229', '21230-21239', '21240-21249', '21250-21259', '21260-21269', '21270-21279', '21280-21289', '21290-21299'], '7/47': ['21400-21409', '21410-21419', '21420-21429', '21430-21439', '21440-21449', '21450-21459', '21460-21469', '21470-21479', '21480-21489', '21490-21499']}, 'ar_repeat_rho0.9': {'0/8': ['20000-20009', '20010-20019', '20020-20029', '20030-20039', '20040-20049', '20050-20059', '20060-20069', '20070-20079', '20080-20089', '20090-20099'], '0/6': ['20200-20209', '20210-20219', '20220-20229', '20230-20239', '20240-20249', '20250-20259', '20260-20269', '20270-20279', '20280-20289', '20290-20299'], '4/26': ['20400-20409', '20410-20419', '20420-20429', '20430-20439', '20440-20449', '20450-20459', '20460-20469', '20470-20479', '20480-20489', '20490-20499'], '4/11': ['20600-20609', '20610-20619', '20620-20629', '20630-20639', '20640-20649', '20650-20659', '20660-20669', '20670-20679', '20680-20689', '20690-20699'], '6/12': ['20800-20809', '20810-20819', '20820-20829', '20830-20839', '20840-20849', '20850-20859', '20860-20869', '20870-20879', '20880-20889', '20890-20899'], '6/8': ['21000-21009', '21010-21019', '21020-21029', '21030-21039', '21040-21049', '21050-21059', '21060-21069', '21070-21079', '21080-21089', '21090-21099'], '7/33': ['21200-21209', '21210-21219', '21220-21229', '21230-21239', '21240-21249', '21250-21259', '21260-21269', '21270-21279', '21280-21289', '21290-21299'], '7/47': ['21400-21409', '21410-21419', '21420-21429', '21430-21439', '21440-21449', '21450-21459', '21460-21469', '21470-21479', '21480-21489', '21490-21499']}, 'scaled_sigma0.5': {'0/8': ['20000-20009', '20010-20019', '20020-20029', '20030-20039', '20040-20049', '20050-20059', '20060-20069', '20070-20079', '20080-20089', '20090-20099'], '0/6': ['20200-20209', '20210-20219', '20220-20229', '20230-20239', '20240-20249', '20250-20259', '20260-20269', '20270-20279', '20280-20289', '20290-20299'], '4/26': ['20400-20409', '20410-20419', '20420-20429', '20430-20439', '20440-20449', '20450-20459', '20460-20469', '20470-20479', '20480-20489', '20490-20499'], '4/11': ['20600-20609', '20610-20619', '20620-20629', '20630-20639', '20640-20649', '20650-20659', '20660-20669', '20670-20679', '20680-20689', '20690-20699'], '6/12': ['20800-20809', '20810-20819', '20820-20829', '20830-20839', '20840-20849', '20850-20859', '20860-20869', '20870-20879', '20880-20889', '20890-20899'], '6/8': ['21000-21009', '21010-21019', '21020-21029', '21030-21039', '21040-21049', '21050-21059', '21060-21069', '21070-21079', '21080-21089', '21090-21099'], '7/33': ['21200-21209', '21210-21219', '21220-21229', '21230-21239', '21240-21249', '21250-21259', '21260-21269', '21270-21279', '21280-21289', '21290-21299'], '7/47': ['21400-21409', '21410-21419', '21420-21429', '21430-21439', '21440-21449', '21450-21459', '21460-21469', '21470-21479', '21480-21489', '21490-21499']}, 'ar_repeat_rho0.5': {'0/8': ['20000-20009', '20010-20019'], '0/6': ['20200-20209', '20210-20219'], '4/26': ['20400-20409', '20410-20419'], '4/11': ['20600-20609', '20610-20619'], '6/12': ['20800-20809', '20810-20819'], '6/8': ['21000-21009', '21010-21019'], '7/33': ['21200-21209', '21210-21219'], '7/47': ['21400-21409', '21410-21419']}, 'ar_repeat_rho0': {'0/8': ['20000-20009', '20010-20019'], '0/6': ['20200-20209', '20210-20219'], '4/26': ['20400-20409', '20410-20419'], '4/11': ['20600-20609', '20610-20619'], '6/12': ['20800-20809', '20810-20819'], '6/8': ['21000-21009', '21010-21019'], '7/33': ['21200-21209', '21210-21219'], '7/47': ['21400-21409', '21410-21419']}, 'scaled_sigma1': {'0/8': ['20000-20009', '20010-20019'], '0/6': ['20200-20209', '20210-20219'], '4/26': ['20400-20409', '20410-20419'], '4/11': ['20600-20609', '20610-20619'], '6/12': ['20800-20809', '20810-20819'], '6/8': ['21000-21009', '21010-21019'], '7/33': ['21200-21209', '21210-21219'], '7/47': ['21400-21409', '21410-21419']}, 'zero': {'0/8': ['20000-20001'], '0/6': ['20200-20201'], '4/26': ['20400-20401'], '4/11': ['20600-20601'], '6/12': ['20800-20801'], '6/8': ['21000-21001'], '7/33': ['21200-21201'], '7/47': ['21400-21401']}}   # label: {"task/state": [ids of each block]}
HARNESS_LABEL, HARNESS_FLAGS = "ar_repeat_rho1", "--family ar_repeat --rho 1"
HARNESS_IDS = {'0/8': '10000-10001', '0/6': '10000,10009', '4/26': '10000,10033', '4/11': '10000,10010', '6/12': '10000,10018', '6/8': '10000,10037', '7/33': '1450,1459', '7/47': '6000,6007'}   # one committed success and one failure per state
OUT_NAME = "step18"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
STOP_AT_H = 11.0                   # no new work after this many hours of the session; running jobs are stopped
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 18 at {REPO_REF}: mechanism M1, 8 states x 7 conditions, 2896 episodes "
      f"in 296 jobs, 16 harness episodes; stop after {STOP_AT_H} h, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
previous = [p for p in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True))
            if os.path.isdir(p) and os.path.exists(f"{p}/repo_commit.txt")]
assert len(previous) <= 1, f"attach only the latest version's output, found {previous}"
for prev in previous:
    print("RESUMING from", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
if not previous:
    print("FRESH START: no earlier output attached. If this is the second session, stop and add it as input.", flush=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 15, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






SEARCH = f"cd {REPO} && {PY} -u noise_reachability/deep_noise_search.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}"
ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step18.py --root {OUT_ROOT}"


def n_ids(spec):
    return sum(int(p.split("-")[1]) - int(p.split("-")[0]) + 1 if "-" in p else 1 for p in spec.split(","))


def block_dir(task, state, label, b):
    return f"{OUT_ROOT}/task_{task}/{label}/state{state}_block{b}"


# this session's index: step 6 appends one line to repo_commit.txt per session (0 = the first)
SESSION = len(Path(OUT_ROOT, "repo_commit.txt").read_text().split()) - 1
print("session", SESSION, flush=True)


def done(path, spec):
    return len(read_csv(path)) >= n_ids(spec)


def run_stage(name, jobs, total, max_parallel=4):
    """True when the stage has no work left: a job that failed stops the run, the deadline leaves work for a next version."""
    print(f"{name}: {len(jobs)} of {total} jobs to run; {hours_left():.1f} h left", flush=True)
    if not jobs:
        return True
    if hours_left() <= 0:
        return False
    codes = run_jobs(jobs, max_parallel=max_parallel)
    if failed(codes):
        fail(f"{name} failed: {failed(codes)}")
        return False
    return all(c == 0 for c in codes.values()) and len(codes) == len(jobs)

## 8. Wrapper checks on each task's real model (a few minutes); stop the run if they fail

In [ ]:
CHECK_DONE = True
for t in TASKS:
    r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step5a.py --task_id {t} --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}",
           check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"check_step5a failed on task {t}")
        CHECK_DONE = False
for t in TASKS:
    if not CHECK_DONE:
        break
    r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_noise_processes.py --task_id {t} --policy_path {POLICY_PATH} "
           f"--n_action_steps {N_ACTION_STEPS}", check=False, env=RUN_ENV)
    print(r.stdout[-1500:], r.stderr[-1500:])
    if r.returncode != 0:
        fail(f"check_noise_processes failed on task {t}")
        CHECK_DONE = False

## 9. Harness, every session: two committed episodes per state, rerun through the new noise-process code; stops the run if it fails

Each state's two episodes must reproduce the committed rows exactly (success and number of steps). In the first session
a state that does not is dropped from every condition, and at least 7 of 8 states must
reproduce. Every later session reruns the harness in its own folder before any episode, and must reproduce every kept
state again (and its package set must equal the first session's), or the run stops: the stack changed.

In [ ]:
HARNESS_DONE, KEPT = False, []
if CHECK_DONE and not FAILURE:
    jobs = {}
    for t, s in STATES:
        out = f"{OUT_ROOT}/harness/session{SESSION}/task{t}_state{s}"
        spec = HARNESS_IDS[f"{t}/{s}"]
        if not done(f"{out}/per_episode_{HARNESS_LABEL}_state{s}.csv", spec):
            jobs[f"harness_task{t}_state{s}"] = (f"{SEARCH} {HARNESS_FLAGS} --task_id {t} --init_state {s} "
                                                 f"--noise_ids {spec} --output_dir {out}", len(jobs) % 2)
    if run_stage("harness", jobs, len(STATES)):
        r = sh(f"{ANALYZE} --gate harness --out {OUT_ROOT}/step18_harness.json", check=False, env=RUN_ENV)
        print(r.stdout, r.stderr[-2000:])
        if r.returncode != 0:
            fail("the harness gate failed: investigate before spending more")
        else:
            KEPT = [tuple(k) for k in json.loads(Path(f"{OUT_ROOT}/step18_harness.json").read_text())["kept"]]
            HARNESS_DONE = True
            print("kept states:", KEPT, flush=True)

## 10. The episodes: block by block (10 ids, one process each), every kept state and condition, four at a time

Blocks run in order, so a session stopped at the deadline leaves the conditions with nearly equal counts. Block 0
holds every condition; blocks 2-9 only `repeat`, rho 0.9 and sigma 0.5.

In [ ]:
EPISODES_DONE = False
if HARNESS_DONE and not FAILURE:
    jobs, total = {}, 0
    for b in range(10):
        for t, s in KEPT:
            for label in CONDITIONS:
                specs = BLOCKS[label][f"{t}/{s}"]
                if b >= len(specs):
                    continue
                total += 1
                out = block_dir(t, s, label, b)
                if not done(f"{out}/per_episode_{label}_state{s}.csv", specs[b]):
                    jobs[f"{label}_task{t}_state{s}_block{b}"] = (
                        f"{SEARCH} {FLAGS[label]} --task_id {t} --init_state {s} --noise_ids {specs[b]} --output_dir {out}",
                        len(jobs) % 2)
    EPISODES_DONE = run_stage("episodes", jobs, total)

## 11. Report (only when every episode is in)

In [ ]:
if EPISODES_DONE and not FAILURE:
    if sh(f"{ANALYZE} --out {OUT_ROOT}/step18_report.txt --table {OUT_ROOT}/step18_table.csv",
          check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")

## 12. Work left?

In [ ]:
if FAILURE:
    print("FAILED:", FAILURE)
elif EPISODES_DONE:
    print("STEP 18 DONE: download step18_results.zip")
else:
    print("WORK REMAINS: add this version's output as input and Save & Run All again")

## 13. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step18_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step18_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)